# Where the pulse starts: the baseline window and `bstd`, channel by channel

*Draft prepared with Claude Code for A. Villano, 2026-11-02.*

`bstd-distribution.ipynb` showed that the baseline noise `bstd` of the real triggers grows with the pulse height (the "ridge" of the height against `bstd`), most clearly in Side2pF. The baseline of the processing is the mean of samples 10 to 500, and the trigger is at sample 500. If the pulse already starts a little before the trigger, its rising edge is in the baseline window and is part of `bstd`. Here that is tested **in every channel**, on the raw traces.

For every real trigger of a channel, from one dump of every series (7 dumps; the series `07221113_1159`, low in every channel, is left out), read from the raw files in `~/idx`:

* `bstd_full`: the standard deviation of samples 10 to 500, the window of the processing;
* `bstd_early`: the same for samples 10 to 400, before the edge;
* the **onset** of the pulse: the first sample at which the trace smoothed over 16 samples rises above 5 times the smoothed noise of samples 10 to 400.

Then, for each channel, the three densities (`bstd_full` and `bstd_early` against the height, the onset), and traces from the ridge and from well above it (`bstd_full` at least 3 times that of the events of the same height). Run with the `darkmatter_cli_env` kernel.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from matplotlib import colors

import goldpan.channels as pch
from goldpan.adapters import idx
from goldpan.config import DCRC_RUN76
from goldpan.dataset import Dataset

LETTERS = "ABCDEF"
CONFIG = DCRC_RUN76                                   # 0.8 us per sample, baseline = samples 10 to 500
RAW = Path.home() / "idx"
_here = Path.cwd().resolve()
PROCESSING = next(p / "data_analysis" / "R76" / "processing" for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
OUT = Path(Dataset.open(PROCESSING / "r76-82v-dcrc1pb.dataset.toml").spec.output)
SENSORS = {c.name: (c.detector, c.channel) for c in pch.sensor_channels()}
g_, p_, SM, N = CONFIG.glitch_samples, CONFIG.pretrigger_samples, 16, 4096
EARLY_END = 400
EXCLUDE = {"07221113_1159"}
DUMP = 10
SATH = np.array([1277, 1095, 589, 930, 680, 883.0])      # saturated height of each channel (saturation-over-time.ipynb)

In [ ]:
def smooth_rows(y):
    c = np.cumsum(np.pad(y, ((0, 0), (1, 0))), axis=1)
    return (c[:, SM:] - c[:, :-SM]) / SM

series_l, event_l = [], []
R = {k: [] for k in ("height", "full", "early", "onset")}                # each: list of (events, 6) blocks
store = {}                                                               # (series, event) -> (6, N) traces, baseline subtracted
for s in sorted({p.name.split("_F")[0] for p in (OUT / "NAI-S2").glob("*_F0010.parquet")} - EXCLUDE):
    t = pq.read_table(OUT / "NAI-S2" / f"{s}_F{DUMP:04d}.parquet", columns=["event", "trigger_type"])
    real_ev = t["event"].to_numpy()[np.asarray(t["trigger_type"].to_pylist()) == "Unknown"]
    e = idx.read(RAW / f"{s}_F{DUMP:04d}", detector_name="NAI-S2", channels=SENSORS, config=CONFIG)
    seg = {name: sg for (name, sg) in e.traces}
    number = np.asarray(e.quantities["event"])
    keep = np.isin(number, real_ev)
    number = number[keep]
    h, full, early, onset, Y = [], [], [], [], []
    for k in LETTERS:
        x = np.asarray(e.traces[(f"Side2p{k}", seg[f"Side2p{k}"])], dtype=np.float32)[keep]
        y = x - x[:, g_:p_].mean(axis=1, keepdims=True)
        ys = smooth_rows(y)
        sig_e = x[:, g_:EARLY_END].std(axis=1) / np.sqrt(SM)
        above = ys > 5 * sig_e[:, None]
        above[:, :g_] = False
        h.append(y.max(axis=1)); full.append(x[:, g_:p_].std(axis=1)); early.append(x[:, g_:EARLY_END].std(axis=1))
        onset.append(np.where(above.any(axis=1), above.argmax(axis=1) + SM // 2, -1))
        Y.append(y)
    for key, v in zip(R, (h, full, early, onset)):
        R[key].append(np.array(v).T)
    series_l += [s] * len(number); event_l += list(number)
    for n, ev_ in enumerate(number):
        store[(s, int(ev_))] = np.array([Y[c][n] for c in range(6)])
R = {k: np.vstack(v) for k, v in R.items()}
SER, EV = np.array(series_l), np.array(event_l)
print(f"{len(EV):,} real triggers in {len(set(series_l))} dumps")
print(f"{'':>34}" + "".join(f"{'Side2p' + k:>10}" for k in LETTERS))
big = R["height"] > 0.15 * SATH                                   # a clear pulse: more than 15 % of the saturated height
print(f"{'events with height > 15 % of sat.':>34}" + "".join(f"{big[:, i].sum():>10,}" for i in range(6)))
print(f"{'onset before sample 500':>34}" + "".join(f"{(R['onset'][big[:, i], i] < p_).mean():>10.1%}" for i in range(6)))
print(f"{'median onset (sample)':>34}" + "".join(f"{np.median(R['onset'][big[:, i], i]):>10.0f}" for i in range(6)))
print(f"{'median bstd_full (10 to 500)':>34}" + "".join(f"{np.median(R['full'][big[:, i], i]):>10.1f}" for i in range(6)))
print(f"{'median bstd_early (10 to 400)':>34}" + "".join(f"{np.median(R['early'][big[:, i], i]):>10.1f}" for i in range(6)))
print(f"{'bstd_early > 5 x its median':>34}" + "".join(
    f"{(R['early'][:, i] > 5 * np.median(R['early'][:, i])).mean():>10.1%}" for i in range(6)))

## The three densities, channel by channel

Left: `bstd_full` (samples 10 to 500) against the height. Middle: `bstd_early` (samples 10 to 400). Right: the onset sample of the pulses above 15 % of the saturated height; the red dotted line is the trigger (sample 500), the grey one the end of the early window. If the pulse starts inside the window, the left panel has the ridge, the middle one does not, and the onset sits just before 500.

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(16, 22))
for i, k in enumerate(LETTERS):
    hb = np.geomspace(3, 1500, 90)
    sb = np.geomspace(1.2, 400, 90)
    for ax, key, title in zip(axes[i, :2], ("full", "early"), ("bstd_full: samples 10 to 500", "bstd_early: samples 10 to 400")):
        ax.hist2d(R["height"][:, i].clip(1), R[key][:, i], bins=[hb, sb], norm=colors.LogNorm(), cmin=1)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("height (counts)"); ax.set_ylabel(f"Side2p{k}: {title}")
    ax = axes[i, 2]
    ax.hist(R["onset"][big[:, i], i], bins=np.arange(0, 800, 5), color="C0")
    ax.axvline(p_, color="C3", ls=":"); ax.axvline(EARLY_END, color="0.5", ls=":")
    ax.set_xlabel("onset of the pulse (sample)"); ax.set_ylabel("events")
    ax.set_title(f"Side2p{k}: height > 15 % of the saturated height", fontsize=9)
fig.tight_layout()
plt.show()

## Traces from the ridge and from above it

Per channel, four heights (5, 15, 40 and 90 % of the saturated height). Top row: an event on the ridge (its `bstd_full` within 15 % of the median of the events of that height). Bottom row: an event with `bstd_full` at least 3 times that median. Raw trace in grey, smoothed in blue, the baseline window shaded, the trigger red, the detected onset green. Change `SEED` for other events.

In [ ]:
SEED = 1
rng = np.random.default_rng(SEED)
tt = np.arange(900)
for i, k in enumerate(LETTERS):
    fig, axes = plt.subplots(2, 4, figsize=(17, 6.5))
    for col, frac in enumerate((0.05, 0.15, 0.40, 0.90)):
        h = frac * SATH[i]
        near = np.abs(np.log(R["height"][:, i].clip(1) / h)) < 0.2
        med = np.median(R["full"][near, i])
        on = np.flatnonzero(near & (np.abs(R["full"][:, i] / med - 1) < 0.15))
        off = np.flatnonzero(near & (R["full"][:, i] > 3 * med))
        for row, pool in enumerate((on, off)):
            ax = axes[row, col]
            if len(pool) == 0:
                ax.axis("off"); continue
            j = int(rng.choice(pool))
            y = store[(SER[j], int(EV[j]))][i]
            ax.axvspan(g_, p_, color="0.9", zorder=0)
            ax.axvline(p_, color="C3", ls=":", lw=0.8)
            ax.plot(tt, y[:900], lw=0.5, color="0.55")
            ys1 = smooth_rows(y[None, :])[0]
            ax.plot(np.arange(900 - SM // 2) + SM // 2, ys1[: 900 - SM // 2], lw=1.1, color="C0")
            if R["onset"][j, i] >= 0:
                ax.axvline(R["onset"][j, i], color="C2", lw=0.9)
            ax.set_title(f"{'ridge' if row == 0 else 'above the ridge'}: height {R['height'][j, i]:.0f}\n"
                         f"bstd full {R['full'][j, i]:.1f}, early {R['early'][j, i]:.1f}, onset {R['onset'][j, i]}", fontsize=9)
            ax.set_xlabel("sample (trigger at 500, red)")
            if col == 0:
                ax.set_ylabel(f"Side2p{k}: counts above the baseline")
    fig.suptitle(f"Side2p{k}", y=1.0)
    fig.tight_layout()
    plt.show()

**What this shows, and what is not done.** For every channel, whether the pulse starts inside the baseline window of the processing, and how `bstd` of the early window compares. Not done: a cut, a change of the baseline window in the processing (`pretrigger_samples` of the readout; it would also change `baseline_mean`, `integral` and `current_max`), and a larger sample than one dump per series.